# GeoCPU：保存済み40,500の評価

新しいColabセッションでも、上から2つのコードセルで準備と評価を実行できます。学習済み40,500のモデルを使います。学習の再実行は含みません。

ColabのGPUランタイムを選び、最初のセルでDriveへの接続を許可してください。既存の固定10k ZIPとチェックポイントをDriveから読みます。

## 1. 準備

ZIPのSHAを確認して新しい作業フォルダーへ展開し、`live_logging.py`など同梱コードの読み込みパスを設定します。このセルではGPUを実行しません。

In [ ]:
# Colab準備セル。保存済みモデルから評価するためのコードだけを復元します。
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import hashlib
import importlib
import sys
import tempfile
import zipfile

STUDY = Path("/content/drive/MyDrive/GeoCPU_v47_fullmodel_20261106")
ARCHIVE = (
    STUDY / "GeoCPU_practical_training_logging_20261107"
    / "GeoCPU_practical_training_logging_20261107_10k.zip"
)
EXPECTED_SHA256 = "939e75d61aba5cd9f271b46e3d210d42af5a0c51904b80df101dbf40072d55d5"

if not ARCHIVE.is_file():
    raise FileNotFoundError(f"実行用ZIPが見つかりません: {ARCHIVE}")
if hashlib.sha256(ARCHIVE.read_bytes()).hexdigest() != EXPECTED_SHA256:
    raise RuntimeError("実行用ZIPのSHA-256が一致しません")

WORK = Path(tempfile.mkdtemp(prefix="geocpu_eval40500_", dir="/content"))
with zipfile.ZipFile(ARCHIVE) as archive:
    archive.extractall(WORK)

# 再実行時も、今回のWORKにある同梱モジュールを読み込みます。
for name in (
    "live_logging", "prepare_training", "frozen_eval",
    "train_candidate", "eval_candidate", "compare_results",
):
    sys.modules.pop(name, None)
sys.path.insert(0, str(WORK))
importlib.invalidate_caches()

from live_logging import run_with_logging
from train_candidate import refuse_active_work
from compare_results import write_reports

print("準備完了:", WORK)
print("続けて評価セルを実行してください。学習のやり直しは不要です。")


## 2. 評価と比較

40,500を先に評価し、30,500の比較対象をそろえます。元T1〜T8・Python補助試験・知識診断を含み、JMMLU7,536問とJamC-QA2,309問は検索なし／検索許可の両条件でも全件評価します。全件知識の追加だけで19,690生成／モデルです。検索は固定ローカルコーパスを使います。

ログはColabとDriveへ保存します。同じ条件の評価結果が保存済みの場合は再利用します。既存の稼働ロックを自動削除しません。

In [ ]:
# 先にcolab_setup.pyの準備セルを実行してください。評価だけを開始します。
from pathlib import Path
import sys
import uuid
import torch
from IPython.display import display, Markdown
from live_logging import run_with_logging
from train_candidate import refuse_active_work
from compare_results import write_reports

if not torch.cuda.is_available():
    raise RuntimeError("ColabのランタイムをGPUに設定してから評価を実行してください")

WORK, STUDY = Path(WORK), Path(STUDY)
BASE_RUN = "v47_from28000_to30500_curriculum_v1"
NEW_RUN = "practical_20261107_10k_multistep_40500"
BASE_RESULTS = STUDY / "practical_20261107_10k_baseline_full_knowledge"
NEW_RESULTS = STUDY / f"{NEW_RUN}_eval_full_knowledge"
ORIGINAL_BASELINE = (
    STUDY / "v47_train_eval_suite_step_30500" / "original_tiers"
)


def evaluate(run, step, output, reuse_original=None):
    refuse_active_work(STUDY)
    locks = []
    for folder in (BASE_RESULTS, NEW_RESULTS):
        locks += list(folder.rglob("logger_active.json"))
        locks += list(folder.rglob("tier_stage_active.json"))
    if locks:
        raise RuntimeError(f"評価の稼働ロックがあります: {locks}")
    command = [
        sys.executable, "-u", str(WORK / "eval_candidate.py"), "run",
        "--study", str(STUDY), "--run", run, "--step", str(step),
        "--output", str(output), "--full-knowledge",
    ]
    if (output / "plan.json").is_file():
        command.append("--resume")
    elif output.exists():
        raise RuntimeError(f"評価計画のない既存フォルダーです: {output}")
    if reuse_original is not None:
        command += ["--reuse-original-from", str(reuse_original)]
    log_dir = (
        STUDY / "practical_20261107_10k_console"
        / f"eval_{step}_{uuid.uuid4().hex[:8]}"
    )
    rc = run_with_logging(command, log_dir, watch_root=output, cwd=WORK)
    if rc:
        raise RuntimeError(f"評価が停止しました。ログ: {log_dir}")
    if not (output / "original_tiers" / "complete.json").is_file():
        raise RuntimeError(f"元T1〜T8が未完了です。ログを確認: {output}")


evaluate(NEW_RUN, 40500, NEW_RESULTS)
evaluate(BASE_RUN, 30500, BASE_RESULTS, ORIGINAL_BASELINE)

REPORT_DIR = STUDY / f"{NEW_RUN}_comparison_{uuid.uuid4().hex[:8]}"
write_reports(BASE_RESULTS, NEW_RESULTS, REPORT_DIR)
display(Markdown((REPORT_DIR / "comparison.md").read_text()))
print("比較CSV:", REPORT_DIR / "comparison.csv")
print("比較JSON:", REPORT_DIR / "comparison.json")
